[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/23_conv2d.ipynb)

# 🟡 Medium: 2D Convolution

*Core Ops & Layers*
Implement a 2-D **convolution** (really a cross-correlation, like every deep
learning framework) from scratch.

$$y[n,p,q,o] = b[o] + \sum_{i=0}^{K_H-1}\sum_{j=0}^{K_W-1}\sum_{c=0}^{C_{in}-1}
x[n,\; p s_h + i,\; q s_w + j,\; c]\; w[i,j,c,o]$$

### Rules
- Signature: `conv2d(x, w, b=None, stride=1, padding="VALID")`
- `x` is **NHWC**: `(N, H, W, C_in)`; `w` is **HWIO**: `(KH, KW, C_in, C_out)`
- `stride` is an int or a `(sh, sw)` pair; `padding` is `"VALID"` or `"SAME"`
- `b` is `(C_out,)` or `None`
- Returns `(N, H_out, W_out, C_out)`
- **Banned**: `jax.lax.conv_general_dilated` (and its `conv`/`conv_with_general_padding`
  wrappers), `jax.scipy.signal.convolve*`, and anything else that does the
  convolution for you
- Must be jittable and differentiable — no Python loop over output pixels, no
  data-dependent control flow

### Output sizes
$$\text{VALID}:\; H_{out} = \left\lfloor\frac{H - K_H}{s_h}\right\rfloor + 1
\qquad
\text{SAME}:\; H_{out} = \left\lceil\frac{H}{s_h}\right\rceil$$

### Why this is the interview question
Three traps live in here.

**1. Layout.** JAX's native layout is NHWC with HWIO kernels — channels last,
output channels last. Frameworks that default to NCHW/OIHW transpose on the way
in, and a silently transposed kernel still produces plausible-looking numbers.
Getting `einsum('nhwijc,ijco->nhwo', ...)` right first try is the actual test.

**2. SAME padding is asymmetric.** If `(out-1)*s + K - H` is odd, XLA puts the
extra row on the **bottom** and the extra column on the **right**. A symmetric
`pad=K//2` gives the same answer only for odd kernels at stride 1 — which is why
the bug survives so long before someone runs a 2-strided even-kernel layer and
gets an off-by-one shift.

**3. im2col is not an optimisation detail.** Materialising the
`(N, H_out, W_out, KH, KW, C_in)` patch tensor blows the input up by $K_H K_W$
and then hands the work to one big matmul — which is exactly what cuDNN does
under the hood for most shapes, because a GEMM on tensor cores beats a bespoke
sliding-window kernel. The memory blow-up is real, though: that tensor is 9x the
input for a 3x3 kernel, so production kernels fuse the gather into the GEMM
rather than writing it out.

Note that this is **cross-correlation** — no kernel flip. Learned kernels make
the distinction irrelevant (the network just learns the flipped filter), so
every framework quietly dropped the flip and kept the name.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def conv2d(x, w, b=None, stride=1, padding="VALID"):
    """2-D cross-correlation.

    Args:
        x:       (N, H, W, C_in)  input, channels last
        w:       (KH, KW, C_in, C_out) kernel
        b:       (C_out,) bias or None
        stride:  int or (sh, sw)
        padding: "VALID" or "SAME"

    Returns:
        (N, H_out, W_out, C_out)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# A 3x3 edge detector on a single-channel step image.
img = jnp.concatenate([jnp.zeros((1, 6, 3, 1)), jnp.ones((1, 6, 3, 1))], axis=2)
sobel = jnp.array([[-1.0, 0.0, 1.0],
                   [-2.0, 0.0, 2.0],
                   [-1.0, 0.0, 1.0]]).reshape(3, 3, 1, 1)

print("VALID:", conv2d(img, sobel, padding="VALID").shape)
print("SAME :", conv2d(img, sobel, padding="SAME").shape)
print("SAME, stride 2:", conv2d(img, sobel, stride=2, padding="SAME").shape)
print(conv2d(img, sobel, padding="SAME")[0, :, :, 0])   # spike at the edge column

# Where the asymmetry shows: H=5, K=2, stride=2 needs 1 pad row -> all of it at the bottom.
x = jnp.arange(25.0).reshape(1, 5, 5, 1)
k = jnp.ones((2, 2, 1, 1))
print("SAME 2x2 stride 2 ->", conv2d(x, k, stride=2, padding="SAME").shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("conv2d")

# hint("conv2d")      # stuck? nudge without the answer
# solution("conv2d")  # spoiler: the reference implementation